In [4]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Завантажуємо навчальний набір даних Iris
iris = load_iris(as_frame=True)

# Перетворюємо його у таблицю Pandas
data = iris.frame

# Переглядаємо перші 5 рядків
print(data.head())

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  


In [5]:
# Розділення на навчальний та тестовий набори
data_train, data_test = train_test_split(data, test_size=0.2, random_state=42)

In [6]:
#Спочатку поділимо дані вручну:

# Перемішуємо дані
data_shuffled = data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# 80 % — навчальні дані
split_index = int(len(data_shuffled) * 0.8)

data_train = data_shuffled.iloc[:split_index]
data_test = data_shuffled.iloc[split_index:]

print("Навчальних прикладів:", len(data_train))
print("Тестових прикладів:", len(data_test))

Навчальних прикладів: 120
Тестових прикладів: 30


In [7]:
#Далі зробимо дуже простий класифікатор. Для кожного класу обчислимо середні характеристики:

feature_columns = [
    'sepal length (cm)',
    'sepal width (cm)',
    'petal length (cm)',
    'petal width (cm)'
]

centroids = data_train.groupby('target')[feature_columns].mean()

centroids

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
target,,,,
0,4.986047,3.434884,1.467442,0.248837
1,5.920513,2.771795,4.264103,1.338462
2,6.607895,2.976316,5.550000,2.047368


In [8]:
#Потім модель буде порівнювати нову квітку з цими трьома «типовими представниками».

#Для прогнозу:

def predict_class(row):

    distances = {}

    for class_id, centroid in centroids.iterrows():

        distance = sum(
            (row[col] - centroid[col]) ** 2
            for col in feature_columns
        )

        distances[class_id] = distance

    return min(distances, key=distances.get)

In [9]:
#І застосовуємо до тестових даних:

predictions = data_test.apply(
    predict_class,
    axis=1
)

print(predictions.head())

120    1
121    0
122    1
123    1
124    0
dtype: int64


In [10]:
#Після цього порівнюємо прогноз із реальною відповіддю:

results = pd.DataFrame({
    'Фактичний клас': data_test['target'],
    'Прогноз моделі': predictions
})

results.head(10)

,Фактичний клас,Прогноз моделі
120,1,1
121,0,0
122,1,1
123,1,1
124,0,0
125,1,1
126,2,2
127,2,2
128,0,0
129,1,1


In [11]:
#І нарешті порахуємо точність без sklearn:

accuracy = (
    results['Фактичний клас']
    ==
    results['Прогноз моделі']
).mean()

print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.90
